# 📘 Módulo 8 — Modelos Lineares para Classificação
## Regressão Logística, Probabilidades e Métricas de Classificação

- 🔵 Conteúdo essencial (IBM‑like)
- 🟣 Conteúdo expandido (Livro Didático)
- 🟠 Conteúdo avançado (opcional)

Este módulo introduz o modelo mais importante da classificação supervisionada:

# **Regressão Logística**

Ela é a ponte entre:

- regressão linear (módulo 6)  
- modelos preditivos (módulo 7)  
- machine learning moderno (módulos 9 e 10)  

---

# 🎯 Objetivos do Módulo 8

Ao final deste módulo, você será capaz de:

- entender o que é classificação  
- entender por que regressão linear não funciona para classes  
- ajustar modelos de regressão logística  
- interpretar coeficientes como **odds ratio**  
- calcular probabilidades  
- gerar previsões de classe  
- avaliar modelos com:
  - matriz de confusão  
  - acurácia  
  - precisão  
  - recall  
  - F1‑score  
  - curva ROC  
  - AUC  
- comparar modelos de classificação  
- construir um pipeline completo de classificação  

Este módulo é a base de todo ML supervisionado de classificação.

# 📚 Índice

0. [Setup — bibliotecas e dados](#setup)
1. [Introdução: o que é classificação?](#intro)
2. [Por que regressão linear não funciona para classes?](#linear_fail)
3. [A função logística (sigmoide)](#sigmoid)
4. [Regressão Logística](#logistic)
5. [Probabilidades, log‑odds e odds ratio](#odds)
6. [Previsão de classes](#predict)
7. [Matriz de confusão e métricas](#metrics)
8. [Curva ROC e AUC](#roc)
9. [Projeto Final — Classificador Completo](#projeto)
10. [Apêndice Matemático Avançado](#apendice)

<a id="setup"></a>
# 0. Setup — bibliotecas e dados

Vamos criar uma variável binária para classificação:

- `high_eval = 1` se eval >= 4.5  
- `high_eval = 0` caso contrário  

Isso transforma o problema em:

> **O professor tem avaliação alta? (sim/não)**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, roc_curve, auc
)

plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
ratings_df = pd.read_csv("/home/moacir/projects/ml/IBM/statistics/data/raw/teachingratings.csv")

ratings_df["high_eval"] = (ratings_df["eval"] >= 4.5).astype(int)
ratings_df[["eval", "high_eval"]].head()

---
# 🔵 Conexão com os módulos anteriores

Nos módulos 6 e 7, você aprendeu:

- regressão linear  
- predição contínua  
- treino/teste  
- validação cruzada  
- regularização  

Agora mudamos o foco:

> **Prever categorias, não valores contínuos.**

Exemplos reais:

- fraude (sim/não)  
- churn (sim/não)  
- doença (sim/não)  
- aprovação de crédito (sim/não)  
- clique em anúncio (sim/não)  

A regressão logística é o modelo mais usado nesses cenários.

<a id="intro"></a>
# 1. Introdução: o que é classificação?

Classificação é o problema de prever **categorias**.

Exemplos:

- spam vs não spam  
- cliente vai cancelar?  
- tumor é maligno?  
- aluno vai passar?  

A saída é:

- 0 ou 1  
- sim ou não  
- positivo ou negativo  

A regressão logística transforma variáveis contínuas em **probabilidades**.

<a id="linear_fail"></a>
# 2. Por que regressão linear NÃO funciona para classificação?

Antes de introduzir a regressão logística, precisamos entender
por que **não podemos usar regressão linear** para prever classes.

Vamos tentar prever:

> **high_eval (0 ou 1)**  

usando regressão linear.

In [ ]:
from sklearn.linear_model import LinearRegression

X = ratings_df[["beauty"]]
y = ratings_df["high_eval"]

lin_model = LinearRegression()
lin_model.fit(X, y)

y_pred_lin = lin_model.predict(X)
y_pred_lin[:10]

---
# 2.1 Problema 1 — A regressão linear gera valores fora do intervalo [0, 1]

A saída da regressão linear é:

$$
\hat{y} = \beta_0 + \beta_1 x
$$

Isso pode gerar:

- valores negativos  
- valores maiores que 1  

Mas probabilidades **devem estar entre 0 e 1**.

In [ ]:
y_pred_lin.min(), y_pred_lin.max()

🟣 **Interpretação**

- valores negativos → impossível como probabilidade  
- valores > 1 → impossível como probabilidade  

Portanto, regressão linear **não pode modelar probabilidades**.

---
# 2.2 Problema 2 — A relação entre X e probabilidade NÃO é linear

Em classificação binária, a probabilidade real geralmente tem forma de S:

baixa probabilidade → transição → alta probabilidade

A regressão linear tenta forçar uma **reta**, o que distorce a relação.

A regressão logística usa a **função sigmoide**, que tem exatamente esse formato.

---
# 2.3 Problema 3 — A regressão linear não modela odds

Em classificação, o que realmente importa é:

- chance de sucesso  
- chance de fracasso  

A razão entre elas é chamada de **odds**:

$$
\text{odds} = \frac{p}{1 - p}
$$

A regressão linear não consegue modelar odds corretamente.

A regressão logística modela:

$$
\log\left(\frac{p}{1 - p}\right)
$$

que é chamado de **log‑odds**.

---
# 2.4 Problema 4 — A regressão linear não maximiza verossimilhança

Em classificação, a função de perda correta é a **entropia cruzada**:

$$
-[y \log(p) + (1-y)\log(1-p)]
$$

A regressão linear usa erro quadrático:

$$
(y - \hat{y})^2
$$

Isso é matematicamente incorreto para variáveis binárias.

A regressão logística usa a função de perda correta.

---
# 2.5 Visualização — Regressão linear tentando prever classes

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(X, y, alpha=0.4, label="Dados reais")
plt.plot(X, y_pred_lin, color="red", label="Regressão Linear")
plt.xlabel("beauty")
plt.ylabel("high_eval")
plt.title("Regressão Linear tentando prever classes")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

- A linha vermelha não respeita os limites 0 e 1  
- A relação não é linear  
- A linha não representa probabilidades  

Isso mostra visualmente por que precisamos da regressão logística.

---
# 2.6 Conclusão desta parte

A regressão linear falha em classificação porque:

✔️ produz valores fora de [0, 1]  
✔️ não modela probabilidades  
✔️ não modela odds  
✔️ não usa a função de perda correta  
✔️ não captura a forma em S da probabilidade  

Por isso precisamos da **regressão logística**, que resolve todos esses problemas.

Agora estamos prontos para:

**PARTE 3 — A Função Logística (Sigmoide).**

<a id="sigmoid"></a>
# 3. A Função Logística (Sigmoide)

Agora que entendemos por que a regressão linear falha em classificação,
vamos introduzir a função que resolve todos esses problemas:

# **A função logística (sigmoide)**

Ela transforma qualquer número real em um valor entre 0 e 1.

A fórmula é:

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

Onde:
- $z$ pode ser qualquer número real  
- $\sigma(z)$ sempre estará entre 0 e 1  

Isso é perfeito para modelar **probabilidades**.

---
# 3.1 Visualizando a função sigmoide

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-10, 10, 200)
s = sigmoid(z)

plt.figure(figsize=(7, 4))
plt.plot(z, s)
plt.title("Função Logística (Sigmoide)")
plt.xlabel("z")
plt.ylabel("σ(z)")
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

- Para valores muito negativos → probabilidade ≈ 0  
- Para valores muito positivos → probabilidade ≈ 1  
- A transição é suave e contínua  

Esse formato em S é ideal para classificação binária.

---
# 3.2 Ligação com regressão logística

A regressão logística calcula:

$$
z = \beta_0 + \beta_1 x_1 + \dots + \beta_k x_k
$$

E depois transforma esse valor em probabilidade usando a sigmoide:

$$
p = \sigma(z)
$$

Ou seja:

> **A regressão logística é apenas regressão linear + sigmoide.**

---
# 3.3 A sigmoide resolve os problemas da regressão linear

✔️ Mantém a saída entre 0 e 1  
✔️ Modela probabilidades corretamente  
✔️ Modela odds e log‑odds  
✔️ Usa a função de perda correta (entropia cruzada)  
✔️ Captura a forma em S da probabilidade  

Por isso ela é usada em:
- regressão logística  
- redes neurais  
- modelos de classificação em geral  

---
# 3.4 Exemplo prático: aplicando sigmoide a valores reais

In [ ]:
test_values = np.array([-5, -1, 0, 1, 5])
sigmoid(test_values)

🟣 **Interpretação**

- valores negativos → probabilidade baixa  
- valores positivos → probabilidade alta  
- zero → probabilidade de 0.5  

Isso faz sentido para classificação binária.

---
# 3.5 Conclusão desta parte

✔️ A sigmoide transforma números reais em probabilidades  
✔️ Ela é a base da regressão logística  
✔️ Ela resolve todos os problemas da regressão linear em classificação  

Agora estamos prontos para:

**PARTE 4 — Regressão Logística (o modelo completo).**

<a id="logistic"></a>
# 4. Regressão Logística — O Modelo Completo

Agora que entendemos a função sigmoide, podemos construir o modelo
que combina:

- regressão linear  
- função logística  

para prever **probabilidades** de uma classe.

A regressão logística modela:

$$
p = \sigma(\beta_0 + \beta_1 x_1 + \dots + \beta_k x_k)
$$

Onde:
- $p$ é a probabilidade de classe 1  
- $\sigma$ é a sigmoide  

Vamos ajustar o modelo para prever:

> **high_eval (1 = avaliação alta, 0 = baixa)**

---
# 4.1 Preparando os dados

In [ ]:
X = ratings_df[["beauty"]]
y = ratings_df["high_eval"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

len(X_train), len(X_test)

---
# 4.2 Ajustando o modelo de Regressão Logística

In [ ]:
log_model = LogisticRegression()
log_model.fit(X_train, y_train)

log_model.coef_, log_model.intercept_

🟣 **Interpretação**

- `coef_` → efeito da variável `beauty` nos **log‑odds**  
- `intercept_` → log‑odds quando beauty = 0  

Esses coeficientes ainda não são probabilidades — vamos interpretar já já.

---
# 4.3 Obtendo probabilidades

A regressão logística sempre retorna **probabilidades** da classe 1.

In [ ]:
y_pred_proba = log_model.predict_proba(X_test)
y_pred_proba[:10]

🟣 **Interpretação**

Cada linha tem duas colunas:

- coluna 0 → probabilidade da classe 0  
- coluna 1 → probabilidade da classe 1  

Exemplo:

`[0.30, 0.70]` → 70% de chance de avaliação alta.

---
# 4.4 Obtendo previsões de classe

O modelo usa um limiar padrão de 0.5:

- p ≥ 0.5 → classe 1  
- p < 0.5 → classe 0  

In [ ]:
y_pred_class = log_model.predict(X_test)
y_pred_class[:10]

---
# 4.5 Interpretando coeficientes: log‑odds

A regressão logística modela:

$$
\log\left(\frac{p}{1-p}\right) = \beta_0 + \beta_1 x
$$

Ou seja:

- os coeficientes são efeitos **nos log‑odds**, não na probabilidade  
- um coeficiente positivo → aumenta a chance da classe 1  
- um coeficiente negativo → reduz a chance da classe 1  

---
# 4.6 Interpretando coeficientes: odds ratio

Para interpretar de forma intuitiva, usamos:

$$
OR = e^{\beta_j}
$$

Onde:
- OR > 1 → aumenta a chance  
- OR < 1 → diminui a chance  

In [ ]:
coef = log_model.coef_[0][0]
odds_ratio = np.exp(coef)
coef, odds_ratio

🟣 **Interpretação**

- Se OR = 1.5 → cada aumento de 1 unidade em beauty aumenta as chances em 50%  
- Se OR = 0.7 → reduz as chances em 30%  

Odds ratio é a forma padrão de interpretar regressão logística.

---
# 4.7 Visualização: probabilidade prevista vs beauty

In [ ]:
beauty_range = np.linspace(X["beauty"].min(), X["beauty"].max(), 200).reshape(-1, 1)
probs = log_model.predict_proba(beauty_range)[:, 1]

plt.figure(figsize=(7, 4))
plt.scatter(X["beauty"], y, alpha=0.3, label="Dados reais")
plt.plot(beauty_range, probs, color="red", label="Probabilidade prevista")
plt.xlabel("beauty")
plt.ylabel("Probabilidade de high_eval = 1")
plt.title("Curva logística ajustada")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

- A curva em S aparece claramente  
- Probabilidade aumenta com beauty  
- A regressão logística captura a transição suave entre 0 e 1  

---
# 4.8 Conclusão desta parte

✔️ Ajustamos um modelo de regressão logística  
✔️ Obtivemos probabilidades e classes  
✔️ Interpretamos coeficientes como log‑odds  
✔️ Convertimos coeficientes em odds ratio  
✔️ Visualizamos a curva logística  

Agora estamos prontos para:

**PARTE 5 — Probabilidades, Log‑Odds e Odds Ratio (interpretação profunda).**

<a id="odds"></a>
# 5. Probabilidades, Log‑Odds e Odds Ratio

Agora que ajustamos a regressão logística, precisamos entender como
interpretar seus coeficientes.

A regressão logística não modela a probabilidade diretamente.

Ela modela:

$$
\log\left(\frac{p}{1 - p}\right)
$$

Esse termo é chamado de:

- **log‑odds**  
- **logit**  

Vamos entender cada parte.

---
# 5.1 Probabilidade

A probabilidade é:

$$
p = P(Y = 1)
$$

A regressão logística retorna essa probabilidade usando a sigmoide:

$$
p = \sigma(z)
$$

onde:

$$
z = \beta_0 + \beta_1 x
$$

---
# 5.2 Odds

As **odds** representam:

> a razão entre a chance de sucesso e a chance de fracasso

$$
\text{odds} = \frac{p}{1 - p}
$$

Exemplos:

- p = 0.5 → odds = 1 (mesma chance)  
- p = 0.75 → odds = 3 (três vezes mais chance de sucesso)  
- p = 0.20 → odds = 0.25 (quatro vezes mais chance de fracasso)  

---
# 5.3 Log‑Odds (Logit)

A regressão logística modela:

$$
\log(\text{odds}) = \beta_0 + \beta_1 x
$$

Ou seja:

- coeficientes são **efeitos lineares nos log‑odds**  
- não são efeitos diretos na probabilidade  

Isso é importante:  
**um coeficiente positivo aumenta os log‑odds, não a probabilidade diretamente.**

---
# 5.4 Odds Ratio — a interpretação mais intuitiva

Para interpretar coeficientes de forma prática, usamos:

$$
OR = e^{\beta_j}
$$

Onde:

- OR > 1 → aumenta as chances  
- OR < 1 → diminui as chances  
- OR = 1 → não muda nada  

Vamos calcular o odds ratio do modelo ajustado.

In [ ]:
coef = log_model.coef_[0][0]
odds_ratio = np.exp(coef)
coef, odds_ratio

🟣 **Interpretação**

- Se OR = 1.30 → cada aumento de 1 unidade em beauty aumenta as chances em 30%  
- Se OR = 0.80 → reduz as chances em 20%  

Odds ratio é a forma padrão de interpretar regressão logística em:
- medicina  
- epidemiologia  
- ciências sociais  
- economia  
- marketing  
- risco de crédito  

---
# 5.5 Convertendo log‑odds em probabilidade

Se o modelo prevê:

$$
z = \beta_0 + \beta_1 x
$$

A probabilidade é:

$$
p = \frac{1}{1 + e^{-z}}
$$

In [ ]:
z_example = log_model.intercept_[0] + coef * 0.5
p_example = 1 / (1 + np.exp(-z_example))
z_example, p_example

🟣 **Interpretação**

- z é o log‑odds  
- p é a probabilidade correspondente  

A sigmoide faz essa conversão automaticamente.

---
# 5.6 Visualização: log‑odds vs probabilidade

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(z, s)
plt.axvline(0, color="red", linestyle="--", label="log-odds = 0")
plt.title("Relação entre log-odds e probabilidade")
plt.xlabel("log-odds (z)")
plt.ylabel("Probabilidade")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

- log‑odds = 0 → probabilidade = 0.5  
- log‑odds > 0 → probabilidade > 0.5  
- log‑odds < 0 → probabilidade < 0.5  

A sigmoide transforma log‑odds em probabilidades de forma suave.

---
# 5.7 Conclusão desta parte

✔️ Probabilidade → valor entre 0 e 1  
✔️ Odds → razão entre sucesso e fracasso  
✔️ Log‑odds → escala linear usada pelo modelo  
✔️ Odds ratio → interpretação intuitiva dos coeficientes  
✔️ A sigmoide converte log‑odds em probabilidades  

Agora estamos prontos para:

**PARTE 6 — Previsão de Classes (thresholds, cutoff, trade‑offs).**

<a id="predict"></a>
# 6. Previsão de Classes — Thresholds e Trade‑offs

A regressão logística retorna **probabilidades**.

Mas, no final, precisamos tomar uma decisão:

> **classe 0 ou classe 1?**

Para isso, usamos um **threshold** (cutoff).

O padrão é:

- p ≥ 0.5 → classe 1  
- p < 0.5 → classe 0  

Mas esse threshold pode (e deve) ser ajustado dependendo do problema.

---
# 6.1 Obtendo probabilidades

In [ ]:
y_proba = log_model.predict_proba(X_test)[:, 1]
y_proba[:10]

🟣 **Interpretação**

Cada valor é a probabilidade prevista de `high_eval = 1`.

---
# 6.2 Threshold padrão (0.5)

In [ ]:
y_pred_default = (y_proba >= 0.5).astype(int)
y_pred_default[:10]

---
# 6.3 Avaliando o threshold padrão

In [ ]:
acc = accuracy_score(y_test, y_pred_default)
prec = precision_score(y_test, y_pred_default)
rec = recall_score(y_test, y_pred_default)
f1 = f1_score(y_test, y_pred_default)

acc, prec, rec, f1

🟣 **Interpretação**

- **Acurácia**: % de previsões corretas  
- **Precisão**: entre os previstos como 1, quantos são realmente 1  
- **Recall**: entre os realmente 1, quantos o modelo encontrou  
- **F1**: média harmônica entre precisão e recall  

O threshold controla o equilíbrio entre precisão e recall.

---
# 6.4 Alterando o threshold

Vamos testar thresholds diferentes:

- 0.3 (mais sensível)  
- 0.5 (padrão)  
- 0.7 (mais conservador)

In [ ]:
def evaluate_threshold(th):
    y_pred = (y_proba >= th).astype(int)
    return {
        "threshold": th,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred)
    }

results = pd.DataFrame([evaluate_threshold(t) for t in [0.3, 0.5, 0.7]])
results

🟣 **Interpretação**

- **Threshold baixo (0.3)** → mais positivos → recall ↑, precisão ↓  
- **Threshold alto (0.7)** → menos positivos → recall ↓, precisão ↑  

Esse é o trade‑off fundamental da classificação.

---
# 6.5 Visualização: impacto do threshold

In [ ]:
thresholds = np.linspace(0.01, 0.99, 50)

precisions = []
recalls = []

for th in thresholds:
    y_pred = (y_proba >= th).astype(int)
    precisions.append(precision_score(y_test, y_pred))
    recalls.append(recall_score(y_test, y_pred))

plt.figure(figsize=(8, 5))
plt.plot(thresholds, precisions, label="Precisão")
plt.plot(thresholds, recalls, label="Recall")
plt.xlabel("Threshold")
plt.ylabel("Métrica")
plt.title("Precisão vs Recall conforme o Threshold")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

- Conforme o threshold sobe → precisão sobe, recall cai  
- Conforme o threshold desce → recall sobe, precisão cai  

Não existe threshold “certo” — depende do problema.

Exemplos:

- **Fraude** → threshold baixo (não pode perder casos)  
- **Spam** → threshold alto (não pode marcar e‑mails bons como spam)  

---
# 6.6 Conclusão desta parte

✔️ A regressão logística retorna probabilidades  
✔️ O threshold define a decisão final  
✔️ Thresholds diferentes mudam precisão e recall  
✔️ Threshold baixo → recall alto, precisão baixa  
✔️ Threshold alto → precisão alta, recall baixo  
✔️ Escolher o threshold depende do contexto do problema  

Agora estamos prontos para:

**PARTE 7 — Matriz de Confusão e Métricas de Classificação.**

<a id="metrics"></a>
# 7. Matriz de Confusão e Métricas de Classificação

Agora que já sabemos:

- como obter probabilidades  
- como transformar probabilidades em classes  
- como ajustar thresholds  

Precisamos avaliar o desempenho do modelo.

A ferramenta central para isso é:

# **A Matriz de Confusão**

---
# 7.1 Matriz de Confusão

A matriz de confusão compara:

- valores reais  
- valores previstos  

Ela contém quatro elementos:

|                | Previsto 0 | Previsto 1 |
|----------------|------------|------------|
| **Real 0**     | TN         | FP         |
| **Real 1**     | FN         | TP         |

Onde:

- **TP**: verdadeiros positivos  
- **TN**: verdadeiros negativos  
- **FP**: falsos positivos  
- **FN**: falsos negativos  

In [ ]:
cm = confusion_matrix(y_test, y_pred_default)
cm

---
# 7.2 Visualização da Matriz de Confusão

In [ ]:
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Previsto")
plt.ylabel("Real")
plt.title("Matriz de Confusão — Threshold 0.5")
plt.show()

🟣 **Interpretação**

- **FP (falso positivo)**: modelo disse que era 1, mas era 0  
- **FN (falso negativo)**: modelo disse que era 0, mas era 1  

Dependendo do problema:

- FP pode ser grave (spam marcando e‑mail bom)  
- FN pode ser grave (não detectar fraude)  

---
# 7.3 Métricas derivadas da matriz de confusão

A partir da matriz, calculamos:

## ✔️ Acurácia
$$
\frac{TP + TN}{TP + TN + FP + FN}
$$

## ✔️ Precisão
$$
\frac{TP}{TP + FP}
$$

## ✔️ Recall (Sensibilidade)
$$
\frac{TP}{TP + FN}
$$

## ✔️ F1‑Score
$$
2 \cdot \frac{\text{Precisão} \cdot \text{Recall}}{\text{Precisão} + \text{Recall}}
$$

In [ ]:
metrics = pd.DataFrame({
    "Acurácia": [accuracy_score(y_test, y_pred_default)],
    "Precisão": [precision_score(y_test, y_pred_default)],
    "Recall": [recall_score(y_test, y_pred_default)],
    "F1": [f1_score(y_test, y_pred_default)]
})

metrics

🟣 **Interpretação**

- **Acurácia** pode enganar quando as classes são desbalanceadas  
- **Precisão** importa quando FP é caro  
- **Recall** importa quando FN é caro  
- **F1** equilibra precisão e recall  

Em muitos problemas reais, **F1 é mais importante que acurácia**.

---
# 7.4 Comparando thresholds usando métricas

Vamos comparar thresholds 0.3, 0.5 e 0.7.

In [ ]:
results

🟣 **Interpretação**

- Threshold baixo → recall alto, precisão baixa  
- Threshold alto → precisão alta, recall baixo  

O threshold ideal depende do contexto.

---
# 7.5 Visualização: Precisão vs Recall

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(thresholds, precisions, label="Precisão")
plt.plot(thresholds, recalls, label="Recall")
plt.xlabel("Threshold")
plt.ylabel("Métrica")
plt.title("Precisão vs Recall conforme o Threshold")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

- A curva mostra o trade‑off fundamental da classificação  
- Não existe threshold “certo” — existe threshold **adequado ao problema**  

---
# 7.6 Conclusão desta parte

✔️ A matriz de confusão é a base da avaliação de classificadores  
✔️ Precisão, recall e F1 vêm diretamente dela  
✔️ Thresholds diferentes mudam o equilíbrio entre FP e FN  
✔️ Acurácia sozinha não é suficiente  
✔️ F1 é útil quando precisamos equilibrar precisão e recall  

Agora estamos prontos para:

**PARTE 8 — Curva ROC e AUC (avaliação independente de threshold).**

<a id="roc"></a>
# 8. Curva ROC e AUC — Avaliação Independente de Threshold

Até agora, avaliamos o modelo usando:

- matriz de confusão  
- precisão  
- recall  
- F1  

Mas todas essas métricas dependem de **um threshold específico**.

A curva ROC permite avaliar o modelo **para todos os thresholds possíveis**.

---
# 8.1 O que é a Curva ROC?

ROC significa:

**Receiver Operating Characteristic**

Ela plota:

- **Eixo X**: FPR (False Positive Rate)  
- **Eixo Y**: TPR (True Positive Rate = Recall)  

Para todos os thresholds entre 0 e 1.

Fórmulas:

$$
TPR = \frac{TP}{TP + FN}
$$

$$
FPR = \frac{FP}{TN + FP}
$$

---
# 8.2 Calculando a curva ROC

In [ ]:
fpr, tpr, thresholds_roc = roc_curve(y_test, y_proba)
fpr[:5], tpr[:5], thresholds_roc[:5]

---
# 8.3 Visualizando a curva ROC

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label="Curva ROC")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Aleatório")
plt.xlabel("FPR (False Positive Rate)")
plt.ylabel("TPR (True Positive Rate)")
plt.title("Curva ROC — Regressão Logística")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

- A diagonal cinza representa um classificador aleatório  
- Quanto mais a curva se afasta da diagonal → melhor o modelo  
- Uma curva perfeita encosta no canto superior esquerdo  

---
# 8.4 AUC — Área Sob a Curva

A métrica mais importante da ROC é:

# **AUC (Area Under the Curve)**

Valores típicos:

- 0.5 → modelo aleatório  
- 0.6–0.7 → fraco  
- 0.7–0.8 → razoável  
- 0.8–0.9 → bom  
- > 0.9 → excelente  

In [ ]:
auc_value = auc(fpr, tpr)
auc_value

🟣 **Interpretação**

- AUC mede a capacidade do modelo de separar as classes  
- É independente de threshold  
- É ideal para comparar modelos  

Quanto maior a AUC, melhor o classificador.

---
# 8.5 Comparando ROC e AUC com outros thresholds

A ROC mostra o desempenho **para todos os thresholds**.

Isso resolve o problema:

- “qual threshold é o melhor?”  

A ROC não escolhe threshold — ela avalia o modelo como um todo.

---
# 8.6 Visualização adicional: TPR vs FPR

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(thresholds_roc, tpr, label="TPR")
plt.plot(thresholds_roc, fpr, label="FPR")
plt.xlabel("Threshold")
plt.ylabel("Taxa")
plt.title("TPR e FPR conforme o Threshold")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

- Threshold baixo → TPR alto, FPR alto  
- Threshold alto → TPR baixo, FPR baixo  

A ROC resume esse trade‑off em uma única curva.

---
# 8.7 Conclusão desta parte

✔️ A curva ROC avalia o modelo para todos os thresholds  
✔️ AUC mede a capacidade de separação entre classes  
✔️ AUC é independente de threshold  
✔️ AUC é ideal para comparar modelos  
✔️ ROC mostra o trade‑off entre TPR e FPR  

Agora estamos prontos para:

**PARTE 9 — Projeto Final: Classificador Completo.**

<a id="projeto"></a>
# 9. Projeto Final — Construindo um Classificador Completo

Neste projeto, vamos aplicar **todo o pipeline de classificação**:

1. Preparação dos dados  
2. Ajuste da regressão logística  
3. Probabilidades e classes  
4. Thresholds  
5. Matriz de confusão  
6. Precisão, recall, F1  
7. Curva ROC  
8. AUC  
9. Interpretação  

O objetivo é prever:

> **high_eval = 1 se eval ≥ 4.5, caso contrário 0**

---
# 9.1 Preparando os dados

In [ ]:
X = ratings_df[["beauty", "age", "students", "female_dummy"]]
y = ratings_df["high_eval"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

len(X_train), len(X_test)

---
# 9.2 Ajustando o modelo

In [ ]:
model = LogisticRegression(max_iter=5000)
model.fit(X_train, y_train)

model.coef_, model.intercept_

---
# 9.3 Probabilidades previstas

In [ ]:
y_proba = model.predict_proba(X_test)[:, 1]
y_proba[:10]

---
# 9.4 Previsões de classe (threshold = 0.5)

In [ ]:
y_pred = (y_proba >= 0.5).astype(int)
y_pred[:10]

---
# 9.5 Matriz de confusão

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm

In [ ]:
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Previsto")
plt.ylabel("Real")
plt.title("Matriz de Confusão — Modelo Final")
plt.show()

---
# 9.6 Métricas de classificação

In [ ]:
metrics = pd.DataFrame({
    "Acurácia": [accuracy_score(y_test, y_pred)],
    "Precisão": [precision_score(y_test, y_pred)],
    "Recall": [recall_score(y_test, y_pred)],
    "F1": [f1_score(y_test, y_pred)]
})

metrics

---
# 9.7 Curva ROC

In [ ]:
fpr, tpr, thresholds_roc = roc_curve(y_test, y_proba)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label="ROC")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Aleatório")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("Curva ROC — Modelo Final")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

---
# 9.8 AUC — Área Sob a Curva

In [ ]:
auc_value = auc(fpr, tpr)
auc_value

---
# 9.9 Interpretando coeficientes (odds ratio)

In [ ]:
odds_ratios = pd.DataFrame({
    "Variável": X.columns,
    "Coeficiente": model.coef_[0],
    "Odds Ratio": np.exp(model.coef_[0])
})

odds_ratios

🟣 **Interpretação**

- OR > 1 → aumenta as chances de high_eval = 1  
- OR < 1 → diminui as chances  
- magnitude → força do efeito  

Isso permite interpretar o modelo de forma clara e prática.

---
# 9.10 Conclusão do Projeto Final

✔️ Construímos um classificador completo  
✔️ Ajustamos regressão logística com múltiplas variáveis  
✔️ Obtivemos probabilidades e classes  
✔️ Avaliamos com matriz de confusão  
✔️ Calculamos precisão, recall e F1  
✔️ Geramos curva ROC  
✔️ Calculamos AUC  
✔️ Interpretamos coeficientes como odds ratio  

Você agora domina **todo o pipeline de classificação supervisionada**.

O próximo módulo aprofunda modelos não lineares:

**Módulo 9 — Árvores de Decisão e Random Forest.**